Sean Bergan

Referring to `Python_exploring_h5ad_files.ipynb` and the [Scanpy Preprocessing and clustering tutorial](https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#nearest-neighbor-graph-construction-and-visualization).

**This notebook includes:**
- Reading in h5ad files in our dataset
- Adding processed layers to our anndata object
- Uploading the resulting .h5ad file to HISE

In [1]:
# Core libraries
import hisepy
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/VertexPartition.py:413: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\frac{1}{m} \\sum_{ij} \\left(A_{ij} - \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/VertexPartition.py:788: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\sum_{ij} \\left(A_{ij} - \\gamma \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/Optimiser.py:27: SyntaxWarning: invalid escape sequence '\g'
  implementation therefore does not guarantee subpartition :math:`\gamma`-density.
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/Optimiser.py:346: SyntaxWarning: invalid escape sequence '\s'
  .. math:: Q = \sum_k \\lambda_k Q_k.


# Read in .h5ad files in our File Set

In [2]:
# todo: store the files as a list for clarity and consistency for referencing when uploading
uuid_list = [
    '02765813-6130-4fac-8708-f01768aa05b6',
    '06b73fab-62d2-4fc3-8a86-2df960cbc1ea',
    '1aecccab-f62a-4c49-b2fe-ba5777930262',
    '207c5f6c-92ec-4690-a7fd-07b0cbebd9f6',
    '3adc31cf-1ea9-4a7d-bc63-31358cb8a325',
    '45e4bd43-4fae-49df-8974-8d043e395f73',
    '4c13d814-1493-48f8-8021-a819b556e97b',
    '56bc5070-2968-4c6b-8198-4e997c75e4fe',
    '64e7735c-e89b-41ef-a293-a39b9ed5ade9',
    '72755c82-880d-4814-8322-6a81ed07466d',
    '9693f71c-dcb3-4d40-b2ad-74fc2ad147de',
    '9f37360e-0191-418b-ab51-fdb86ab9be09',
    'a3bc4704-5fe3-4a74-bce5-0700689db1f6',
    'af42c180-0218-4918-ae12-0a4f43c4aa1f',
    'b790716c-b2de-4028-ad60-1e5a7b81f05d',
    'bfa4c2f1-69d5-4bcf-a1a6-9beb69dbffc9',
    'd076758f-3d76-49b7-91be-0217eefcdf26',
    'd2f8da49-85eb-4dea-a166-5308bb73de91'
]

file_list = hisepy.cache_files(uuid_list)

In [3]:
# view the first five file names
file_list[0:5]

['/home/workspace/input/1742749117/2025_bgmp/02765813-6130-4fac-8708-f01768aa05b6/rhodium-niobium-silver/DMSO_Control_counts_filtered_labeled_sampled.h5ad',
 '/home/workspace/input/1742749117/2025_bgmp/06b73fab-62d2-4fc3-8a86-2df960cbc1ea/rhodium-niobium-silver/Ruxolitinib_INCB018424_counts_filtered_labeled_sampled.h5ad',
 '/home/workspace/input/1742749117/2025_bgmp/1aecccab-f62a-4c49-b2fe-ba5777930262/rhodium-niobium-silver/Canertinib_CI-1033_counts_filtered_labeled_sampled.h5ad',
 '/home/workspace/input/1742749117/2025_bgmp/207c5f6c-92ec-4690-a7fd-07b0cbebd9f6/rhodium-niobium-silver/Baricitinib_LY3009104_INCB028050_counts_filtered_labeled_sampled.h5ad',
 '/home/workspace/input/1742749117/2025_bgmp/3adc31cf-1ea9-4a7d-bc63-31358cb8a325/rhodium-niobium-silver/Erlotinib_counts_filtered_labeled_sampled.h5ad']

In [4]:
# initialize a list
adata_list = []

# iterate over the list h5ad_files to use Scanpy to read the file
for h5ad_file in file_list:
    adata = sc.read_h5ad(h5ad_file)
    # append anndata file object into the list
    adata_list.append(adata)

In [5]:
# view the first item of the list
adata_list[0]

AnnData object with n_obs × n_vars = 25000 × 1916
    obs: 'original_barcodes', 'batch_id', 'pool_id', 'chip_id', 'well_id', 'n_umis', 'n_genes', 'plate_location', 'cyto_treatment', 'drug_treatment', 'drug_name', 'drug_cas_number', 'drug_mw', 'drug_solvent', 'drug_pathway', 'drug_target', 'drug_description', 'drug_chembl_name', 'drug_chembl_id', 'AIFI_L1', 'AIFI_L2', 'leiden_2'
    uns: 'leiden_2', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'

In [6]:
# using anndata to concatenate: code chunk from Scanpy tutorial

# combine multiple anndata objects and adds a "sample" column telling you which dataset each cell came from; aka, which DRUG NAME each cell was treated with!
adata = ad.concat(adata_list, label="sample")

# make sure all observations are unique
adata.obs_names_make_unique()

# filtering out CD8aa because there aren't enough
adata = adata[adata.obs.AIFI_L2 != "CD8aa"]

/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/anndata/_core/merge.py:1667: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat_annot = pd.concat(


## Cleaning up labels
Removing accession numbers/ids from the drug names for the sake of presentation figures

In [7]:
import re

unique_drug_names = adata.obs['drug_name'].unique()

def clean_name(name):
    name = re.sub(' \\(.*\\) ', ' ', name)
    name = re.sub(' \\(.*\\)$', '', name)
    # also fix IL6
    name = re.sub('IL6', 'IL-6', name)
    return name

# for name in unique_drug_names:
#     print(clean_name(name))

adata.obs['clean_drug_name'] = adata.obs['drug_name'].copy().apply(clean_name)

# .map(clean_mapping)

/tmp/ipykernel_1114/3858086046.py:15: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs['clean_drug_name'] = adata.obs['drug_name'].copy().apply(clean_name)


## Normalization

In [8]:
# help(adata)
adata.layers["counts"] = adata.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(adata, target_sum = 1e4)


adata.layers["log_transformed"] = np.log1p(adata.X)
# Logarithmize the data
sc.pp.log1p(adata)
# sc.pp.scale(adata)

## Making the .h5ad file

In [9]:
adata.write_h5ad(filename = "processed_adata.h5ad")

## Uploading to HISE

In [10]:
# starting off with a function to make unique destination strings with periodic table elements
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
        
    rand_str = '-'.join(rand_el)
    return rand_str

In [11]:
help(hisepy.upload_files)

Help on function upload_files in module hisepy.upload:

upload_files(
    files: list,
    study_space_id: str = None,
    project: str = None,
    title: str = None,
    input_file_ids: list = [],
    input_sample_ids: list = [],
    file_types: list = [],
    store: str = None,
    destination: str = '',
    do_prompt: bool = True,
    do_conda_build_check=True
)
    Uploads files to a store and records their provenance in HISE, but V3

    Parameters:
        files (list): absolute filepath of file to be uploaded
        study_space_id (str): ID that pertains to a study in the collaboration space (optional)
        project (str): project short name (required if study space is not specified, defaults to the ide's default setting
        title (str): 10+ character title for upload result
        input_file_ids (list): fileIds from HISE that were utilized to generate a user's result
        input_sample_ids (list): sampleIds from HISE that were utilized to generate a user's result
    

In [12]:
# need unique title for upload, using UTC time
from datetime import datetime, timezone

utc_current = datetime.now(timezone.utc)
print(utc_current)

# todo: should probably clean up the names here of these csvs... just replace "." with "_" in the threshold?
files_to_upload = [
    "processed_adata.h5ad"
]

hisepy.upload_files(
    files = files_to_upload,
    study_space_id = "c8a94b84-b0b7-40a9-980b-81a63ad6e115",
    title = f"processed_h5ad_{utc_current}",
    input_file_ids = uuid_list,
    destination = element_id()
)

2026-01-21 02:20:14.737268+00:00
checking if conda environment can compile...
creating temp conda environment...
temp conda environment created successfully, now packing...


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': 'b06642f6-ed5f-42a2-9663-9a9b0db741bf',
 'ProcessId': 'b7447020-8d16-4eaf-911d-1fd2c7373a68',
 'WorkflowId': 'a48f699f-db4e-4bfc-8e96-f45f64179667',
 'FileIds': ['c9cf5bd2-a3a4-4c11-bd4e-ceea1205ec78']}